In [ ]:
import pandas as pd
import openpyxl
import numpy as np
import re
import unicodedata
from pathlib import Path

# =========================
# 1) Configuração
# =========================
raiz_projeto = Path.cwd()
if not (raiz_projeto / "dados").exists():
    raiz_projeto = raiz_projeto.parent

arquivo_entrada = raiz_projeto / "dados" / "geral_15mai.xlsx"
arquivo_saida = raiz_projeto / "dados" / "analise_duplicadas_geral_15mai.xlsx"

# Critério de duplicidade (colunas do relatório original):
# D, E, H, I, J, K, N, O, Q, S, T
alvos = {
    "nome_apelido": ["nome (apelido)", "nome/apelido", "nome"],
    "objeto": ["objeto"],
    "tipo_natureza": ["tipo (natureza)", "tipo natureza", "natureza", "tipo"],
    "especie": ["espécie", "especie"],
    "investimento_previsto": ["investimento previsto", "investimento"],
    "executor_obra": ["executor da obra", "executor"],
    "situacao_intervencao": ["situação da intervenção", "situacao da intervencao", "situação", "situacao"],
    "sistema": ["sistema"],
    "uf_principal": ["uf (principal)", "uf principal"],
    "municipio": ["município", "municipio"],
    "coordenadas": ["localização (coordenadas)", "localizacao (coordenadas)", "coordenadas", "localização", "localizacao"],
}

# =========================
# 2) Funções auxiliares
# =========================
def norm_str(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s

def norm_text_value(v):
    if pd.isna(v):
        return ""
    v = str(v).strip()
    v = unicodedata.normalize("NFKD", v)
    v = "".join(ch for ch in v if not unicodedata.combining(ch))
    v = v.upper()
    v = re.sub(r"\s+", " ", v)
    return v

def parse_num_br(v):
    if pd.isna(v) or str(v).strip() == "":
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)
    s = str(v).strip()
    s = s.replace(".", "").replace(",", ".")
    s = re.sub(r"[^0-9\.\-]", "", s)
    try:
        return float(s)
    except:
        return np.nan

def detectar_colunas(df_cols, alvos_dict):
    col_norm = {c: norm_str(c) for c in df_cols}
    mapeamento = {}
    usados = set()

    for chave, aliases in alvos_dict.items():
        aliases_norm = [norm_str(a) for a in aliases]
        encontrado = None

        # Match exato por alias normalizado
        for c, cn in col_norm.items():
            if c in usados:
                continue
            if cn in aliases_norm:
                encontrado = c
                break

        # Fallback: contains
        if encontrado is None:
            for c, cn in col_norm.items():
                if c in usados:
                    continue
                if any(a in cn for a in aliases_norm):
                    encontrado = c
                    break

        if encontrado is None:
            raise ValueError(
                f"Não foi possível localizar a coluna para '{chave}'. "
                f"Colunas disponíveis: {list(df_cols)}"
            )

        mapeamento[chave] = encontrado
        usados.add(encontrado)

    return mapeamento

def fmt_br(x, casas=2):
    if pd.isna(x):
        return ""
    s = f"{x:,.{casas}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

# =========================
# 3) Leitura e mapeamento
# =========================
df = pd.read_excel(arquivo_entrada)
map_cols = detectar_colunas(df.columns, alvos)

# Dataframe somente com colunas do critério
base = df[[map_cols[k] for k in map_cols]].copy()
base.columns = list(map_cols.keys())

# =========================
# 4) Normalização para chave de duplicidade
# =========================
norm = pd.DataFrame(index=base.index)

for c in base.columns:
    if c == "investimento_previsto":
        norm[c] = base[c].apply(parse_num_br).round(2).fillna(-999999999).astype(str)
    else:
        norm[c] = base[c].apply(norm_text_value)

# Chave composta
chave = norm.astype(str).agg(" || ".join, axis=1)
df_aux = df.copy()
df_aux["_dup_key"] = chave

# Linhas em duplicidade (considerando todas as ocorrências dos grupos duplicados)
mask_dup = df_aux["_dup_key"].duplicated(keep=False)
dups = df_aux[mask_dup].copy()

# Tamanhos dos grupos
group_sizes = dups["_dup_key"].value_counts()
group_sizes = group_sizes[group_sizes > 1].sort_values(ascending=False)

# =========================
# 5) Métricas principais
# =========================
total_registros = len(df_aux)
total_grupos_dup = int(len(group_sizes))
total_linhas_dup = int(mask_dup.sum())
pct_duplicadas_base = (total_linhas_dup / total_registros * 100) if total_registros else 0.0
taxa_unicidade = 100 - pct_duplicadas_base if total_registros else 0.0
media_itens_grupo = float(group_sizes.mean()) if total_grupos_dup else 0.0
maior_grupo = int(group_sizes.max()) if total_grupos_dup else 0
grupos_apenas_2 = int((group_sizes == 2).sum()) if total_grupos_dup else 0
pct_grupos_apenas_2 = (grupos_apenas_2 / total_grupos_dup * 100) if total_grupos_dup else 0.0

resumo = pd.DataFrame({
    "Indicador": [
        "Total de Registros",
        "Total de Grupos de Duplicatas",
        "Total de Linhas Duplicadas",
        "% de Duplicatas na Base",
        "Taxa de Unicidade",
        "Média de Itens por Grupo",
        "Maior Grupo (mais duplicatas)",
        "Grupos com apenas 2 itens",
        "% de grupos com apenas 2 itens",
    ],
    "Valor": [
        total_registros,
        total_grupos_dup,
        total_linhas_dup,
        pct_duplicadas_base,
        taxa_unicidade,
        media_itens_grupo,
        maior_grupo,
        grupos_apenas_2,
        pct_grupos_apenas_2,
    ],
})

# =========================
# 6) Detalhamento dos grupos
# =========================
grupos = group_sizes.rename("qtd").reset_index().rename(columns={"index": "_dup_key"})
grupos["grupo_id"] = [f"G{i:05d}" for i in range(1, len(grupos) + 1)]

# Anexa grupo_id nas linhas duplicadas
map_gid = dict(zip(grupos["_dup_key"], grupos["grupo_id"]))
dups["grupo_id"] = dups["_dup_key"].map(map_gid)

# Organiza colunas de saída
colunas_saida = ["grupo_id"] + [map_cols[k] for k in map_cols] + ["_dup_key"]
dups_saida = dups[colunas_saida].sort_values(["grupo_id"])

# =========================
# 7) Export e impressão
# =========================
# with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
#     resumo.to_excel(writer, sheet_name="resumo_duplicatas", index=False)
#     grupos[["grupo_id", "qtd", "_dup_key"]].to_excel(writer, sheet_name="grupos_duplicatas", index=False)
#     dups_saida.to_excel(writer, sheet_name="linhas_duplicadas", index=False)

print("=== Resumo da Análise de Duplicatas ===")
print(f"Total de Registros: {total_registros}")
print(f"Total de Grupos de Duplicatas: {total_grupos_dup}")
print(f"Total de Linhas Duplicadas: {total_linhas_dup}")
print(f"% de Duplicatas na Base: {fmt_br(pct_duplicadas_base)}%")
print(f"Taxa de Unicidade: {fmt_br(taxa_unicidade)}%")
print(f"Média de Itens por Grupo: {fmt_br(media_itens_grupo)}")
print(f"Maior Grupo (mais duplicatas): {maior_grupo}")
print(f"Grupos com apenas 2 itens: {grupos_apenas_2} ({fmt_br(pct_grupos_apenas_2)}%)")
# print(f"\nArquivo gerado: {arquivo_saida}")

print("\nMapeamento de colunas utilizado:")
for k, v in map_cols.items():
    print(f"- {k}: {v}")

=== Resumo da Análise de Duplicatas ===
Total de Registros: 125250
Total de Grupos de Duplicatas: 2559
Total de Linhas Duplicadas: 6241
% de Duplicatas na Base: 4,98%
Taxa de Unicidade: 95,02%
Média de Itens por Grupo: 2,44
Maior Grupo (mais duplicatas): 44
Grupos com apenas 2 itens: 2224 (86,91%)

Arquivo gerado: /home/daianasales/vscode/cgimo/exploracao_cgimo/dados/analise_duplicadas_geral_15mai.xlsx

Mapeamento de colunas utilizado:
- nome_apelido: Nome ( Apelido )
- objeto: Objeto
- tipo_natureza: Tipo (Natureza da Intervenção)
- especie: Espécie
- investimento_previsto: Investimento Previsto
- executor_obra: Executor da Obra
- situacao_intervencao: Situação da Intervenção
- sistema: Sistema
- uf_principal: UF (Principal)
- municipio: Município
- coordenadas: Localização (Coordenadas)
